In [ ]:
# ============================================================
# MULTI-PDF CHATBOT USING GROQ + TF-IDF
# No Chroma, no LangChain, no sentence-transformers
# ============================================================

# Install once if needed:
# !pip install -U groq pdfplumber scikit-learn

import os
import pdfplumber
from groq import Groq
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# CONFIGURATION
# ============================================================

PDF_FOLDER = r"C:\Users\poulo\Documents\Documents\PhD\Spring 2026\Infrastructure Failure"

os.environ["GROQ_API_KEY"] = "******************"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

chunks = []
metadata = []
vectorizer = None
tfidf_matrix = None
chat_history = []

# Groq model
GROQ_MODEL = "llama-3.1-8b-instant"
# For better answers, you can try:
# GROQ_MODEL = "llama-3.1-70b-versatile"


# ============================================================
# CHUNK TEXT
# ============================================================

def chunk_text(text, chunk_size=400, overlap=50):
    words = text.split()
    result = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        result.append(" ".join(words[start:end]))
        start += chunk_size - overlap

    return result


# ============================================================
# LOAD ALL PDF FILES
# ============================================================

def load_pdfs_from_folder(pdf_folder):
    global chunks, metadata, vectorizer, tfidf_matrix

    chunks = []
    metadata = []

    pdf_files = [
        f for f in os.listdir(pdf_folder)
        if f.lower().endswith(".pdf")
    ]

    if not pdf_files:
        return "No PDF files found in the folder."

    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_folder, pdf_file)

        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page_num, page in enumerate(pdf.pages, start=1):
                    text = page.extract_text()

                    if text and text.strip():
                        page_chunks = chunk_text(text)

                        for c in page_chunks:
                            chunks.append(c)
                            metadata.append({
                                "file": pdf_file,
                                "page": page_num
                            })

        except Exception as e:
            print(f"Could not read {pdf_file}: {e}")

    if not chunks:
        return "No text could be extracted from the PDFs."

    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(chunks)

    return f"Loaded {len(pdf_files)} PDFs and created {len(chunks)} text chunks."


# ============================================================
# RETRIEVE RELEVANT PDF CHUNKS
# ============================================================

def retrieve_relevant_chunks(question, k=4, max_context_chars=3500):
    global vectorizer, tfidf_matrix, chunks, metadata

    question_vec = vectorizer.transform([question])
    similarities = cosine_similarity(question_vec, tfidf_matrix).flatten()

    top_indices = similarities.argsort()[-k:][::-1]

    context = ""
    sources = []

    for idx in top_indices:
        chunk = chunks[idx]
        source = metadata[idx]

        block = f"[File: {source['file']}, Page: {source['page']}]\n{chunk}\n\n"

        if len(context) + len(block) > max_context_chars:
            break

        context += block
        sources.append(source)

    return context, sources


# ============================================================
# CHAT WITH YOUR PDF LIBRARY
# ============================================================

def chat_with_pdfs(question, k=4):
    global chat_history

    if tfidf_matrix is None:
        return "Please run load_pdfs_from_folder(PDF_FOLDER) first."

    context, sources = retrieve_relevant_chunks(question, k=k)

    recent_history = chat_history[-6:]

    history_text = ""
    for item in recent_history:
        history_text += f"User: {item['user']}\nAssistant: {item['assistant']}\n\n"

    prompt = f"""
You are a research assistant answering questions from a collection of PDF papers.

Use ONLY the PDF context below.
If the answer is not found in the context, say:
"The uploaded PDF collection does not provide enough information."

Conversation history:
{history_text}

PDF Context:
{context}

User question:
{question}

Answer clearly and academically.
Include the source paper/page when possible.
"""

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful research assistant. Answer only from the provided PDF context."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=700
    )

    answer = response.choices[0].message.content

    chat_history.append({
        "user": question,
        "assistant": answer
    })

    source_text = "\n\nSources:\n"
    for i, s in enumerate(sources, start=1):
        source_text += f"{i}. {s['file']} | Page {s['page']}\n"

    return answer + source_text


# ============================================================
# LOAD PDF LIBRARY
# ============================================================

print(load_pdfs_from_folder(PDF_FOLDER))

Loaded 27 PDFs and created 769 text chunks.


In [15]:
!pip install gradio


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\poulo\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [7]:
print(chat_with_pdfs("Compare the methods used for cascading failure modeling."))

According to the provided PDF context, traditional methods for cascading failure modeling, such as state enumeration (SE), have been employed (2.1.2. Resilience assessment method considering the uncertainty of cascading failure processes). However, these methods are limited in their ability to accurately capture the complexities of cascading failure processes, particularly in the context of wildfire disasters (2.1. Basic theory of resilience assessment for cascading failure coupling).

In contrast, more advanced methods have been proposed to analyze the breakdown mechanism of insulation gaps exposed to flames and to evaluate resilience curves (2.1.1. Traditional resilience assessment methods). For instance, models have been developed to analyze the breakdown mechanism of insulation gaps (Li et al., 2016) and to evaluate resilience curves (Zhou et al., 2024). Additionally, adjustments to breakdown voltage for ground and phase insulation have been proposed (Fonseca et al., 1990) to allow

In [16]:
# !pip install -U gradio groq pdfplumber scikit-learn

import os
import pdfplumber
import gradio as gr

from groq import Groq
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PDF_FOLDER = r"C:\Users\poulo\Documents\Documents\PhD\Spring 2026\Infrastructure Failure"

os.environ["GROQ_API_KEY"] = "YOUR_NEW_GROQ_API_KEY"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

GROQ_MODEL = "llama-3.1-8b-instant"

chunks = []
metadata = []
vectorizer = None
tfidf_matrix = None


def chunk_text(text, chunk_size=400, overlap=50):
    words = text.split()
    result = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        result.append(" ".join(words[start:end]))
        start += chunk_size - overlap

    return result


def load_pdfs():
    global chunks, metadata, vectorizer, tfidf_matrix

    chunks = []
    metadata = []

    if not os.path.exists(PDF_FOLDER):
        return f"Folder not found: {PDF_FOLDER}"

    pdf_files = [f for f in os.listdir(PDF_FOLDER) if f.lower().endswith(".pdf")]

    if not pdf_files:
        return "No PDF files found."

    for pdf_file in pdf_files:
        pdf_path = os.path.join(PDF_FOLDER, pdf_file)

        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page_num, page in enumerate(pdf.pages, start=1):
                    text = page.extract_text()

                    if text and text.strip():
                        for c in chunk_text(text):
                            chunks.append(c)
                            metadata.append({
                                "file": pdf_file,
                                "page": page_num
                            })

        except Exception as e:
            print(f"Could not read {pdf_file}: {e}")

    if not chunks:
        return "No text could be extracted from the PDFs."

    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(chunks)

    return f"Loaded {len(pdf_files)} PDFs and {len(chunks)} chunks."


def retrieve(question, k=3, max_context_chars=3000):
    q_vec = vectorizer.transform([question])
    sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_idx = sims.argsort()[-k:][::-1]

    context = ""
    sources = []

    for idx in top_idx:
        block = (
            f"[File: {metadata[idx]['file']}, Page: {metadata[idx]['page']}]\n"
            f"{chunks[idx]}\n\n"
        )

        if len(context) + len(block) > max_context_chars:
            break

        context += block
        sources.append(metadata[idx])

    return context, sources


def answer_question(question, history):
    if tfidf_matrix is None:
        return "Please click **Load PDFs** first."

    context, sources = retrieve(question)

    prompt = f"""
Use only the PDF context below to answer.

PDF Context:
{context}

Question:
{question}

Answer clearly and include source page references.
"""

    try:
        response = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You answer only from the provided PDF context."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            max_tokens=600
        )

        answer = response.choices[0].message.content

        source_text = "\n\nSources:\n"
        for i, s in enumerate(sources, start=1):
            source_text += f"{i}. {s['file']} | Page {s['page']}\n"

        return answer + source_text

    except Exception as e:
        return f"Error: {e}"


with gr.Blocks() as demo:
    gr.Markdown("# Multi-PDF Chatbot")

    load_btn = gr.Button("Load PDFs")
    status = gr.Textbox(label="Status", interactive=False)

    chatbot = gr.Chatbot(label="PDF Chatbot", type="messages")

    question = gr.Textbox(
        label="Ask a question",
        placeholder="Type your question here and press Enter..."
    )

    clear = gr.Button("Clear Chat")

    def user_submit(user_message, chat_history):
        if chat_history is None:
            chat_history = []

        chat_history.append({
            "role": "user",
            "content": user_message
        })

        bot_reply = answer_question(user_message, chat_history)

        chat_history.append({
            "role": "assistant",
            "content": bot_reply
        })

        return "", chat_history

    load_btn.click(load_pdfs, outputs=status)

    question.submit(
        user_submit,
        inputs=[question, chatbot],
        outputs=[question, chatbot]
    )

    clear.click(lambda: [], outputs=chatbot)

demo.launch()

ModuleNotFoundError: No module named 'gradio'